# PEFT Ablation Study — Colab training runner

Runs ONE training method (full fine-tune, one LoRA rank, or QLoRA) end-to-end on a Colab GPU:
trains under the project's fixed compute budget, evaluates the resulting checkpoint (task EM/F1 +
MMLU forgetting benchmark), and pushes the small `results/*.json` files back to GitHub.

This notebook contains **no training logic of its own** — it just clones the repo and runs the
same `scripts/train.py` / `scripts/evaluate.py` already validated locally and on Colab.

**Every run of this notebook is self-contained**: it always deletes and re-clones the repo from
scratch (cell 4), so there's no stale-state ambiguity if a previous attempt partially failed or
the runtime restarted. **If anything errors: `Runtime > Restart session`, then `Runtime > Run all`
from the top — don't try to resume from the middle of a broken session.**

**Before running:**
1. `Runtime > Change runtime type > T4 GPU` (must be a GPU runtime, not CPU).
2. Add a GitHub token via the **Secrets** panel (key/padlock icon in the left sidebar):
   name it `GITHUB_TOKEN`, enable notebook access. Use a **fine-grained PAT scoped only to the
   `peft-ablation-study` repo** with Contents: Read/Write.
3. Set `CONFIG` in the cell below to the method you're running this time.
4. `Runtime > Run all`.

In [ ]:
# <-- CHANGE THIS for each of the 6 runs across Steps 4-6:
CONFIG = "configs/full_finetune.yaml"

# Options across the whole project:
#   configs/full_finetune.yaml   (Step 4)
#   configs/lora_rank4.yaml      (Step 5)
#   configs/lora_rank8.yaml      (Step 5)
#   configs/lora_rank16.yaml     (Step 5)
#   configs/lora_rank32.yaml     (Step 5)
#   configs/qlora.yaml           (Step 6)

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime > Change runtime type > select 'T4 GPU', then Runtime > Run all."
)
print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## Clone (always fresh)

Deletes any existing `/content/repo` first — every run of this notebook starts from a clean,
known-good clone of `main`, never a possibly-stale leftover from an earlier attempt. Prints the
exact commit hash so you can always see which code version actually ran (paste that hash to me if
something still looks wrong — first thing I'll check).

In [ ]:
import os
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/Liang714/peft-ablation-study.git"
REPO_DIR = "/content/repo"

!rm -rf {REPO_DIR}
!git clone --quiet {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)

!echo "HEAD commit:" && git log -1 --format='%H %s'
!echo && echo "Working directory:" && pwd

## Verify required data files are present

Fails fast with a clear message here rather than several confusing steps later (this caught a real
bug once already: `data/splits/*.jsonl` wasn't committed to git, which surfaced as a `FileNotFoundError`
during training that then cascaded into an unrelated-looking Hugging Face 401 error during evaluation).

In [ ]:
from pathlib import Path

required_files = [
    "data/splits/train.jsonl",
    "data/splits/val.jsonl",
    "data/splits/test.jsonl",
    "data/processed/dataset_stats.json",
    "data/processed/mmlu_subset.jsonl",
    CONFIG,
]
missing = [f for f in required_files if not Path(f).exists()]
assert not missing, (
    f"Missing required file(s): {missing}. This means the repo clone is incomplete or a file that "
    "should be committed to git isn't — stop here and report this back rather than continuing."
)
print("All required files present:")
for f in required_files:
    print(" ✓", f)

In [ ]:
# Every package here is pinned to a specific, locally-validated version (not just a loose lower
# bound) — three separate Colab-only bugs earlier all traced back to unpinned versions drifting
# from what was actually tested. Installed as SEPARATE commands, not combined with multiple -r
# flags on one command (pip resolves multiple -r files given to a single command jointly, which
# hits an unresolvable mlflow-vs-datasets pyarrow version conflict — verified this exact
# separate-commands order works):
!pip install -q -r requirements-tracking.txt
!pip install -q -r requirements.txt
if "qlora" in CONFIG:
    !pip install -q -r requirements-gpu.txt

# Colab already ships a CUDA-linked torch satisfying requirements.txt's torch>=2.1, so the above
# should mostly skip reinstalling it. pip may print dependency-conflict WARNINGS — about mlflow's
# pyarrow constraint (expected, see requirements.txt's comment, verified harmless) and about
# unrelated preinstalled Colab packages (rasterio, jax, opencv, etc. wanting numpy>=2, we don't
# use any of those packages) — both are safe to ignore; only a real ERROR/non-zero exit matters.
import importlib
import torch
importlib.reload(torch)
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "CUDA became unavailable after pip install — Runtime > Restart session, then Run all again."

## Train

Runs under the project's fixed compute budget (`configs/base.yaml`'s `compute_budget.total_tokens`,
currently 2,383,278 tokens = 3 epochs over the real train split) — identical token budget for every
method, which is the entire point of this comparison. Logs training wall-clock time and peak GPU
memory, saves the checkpoint to `checkpoints/<run_name>/` (gitignored — stays in this Colab session's
storage, download it manually below if you want to keep it beyond this session).

In [ ]:
!PYTHONPATH=. python scripts/train.py --config {CONFIG}

In [ ]:
# Fail fast with a clear message if training didn't actually produce a checkpoint, instead of
# letting the next cell fail with a confusing error.
import json

from src.config import load_config

cfg = load_config(CONFIG)
RUN_NAME = cfg.run_name
CHECKPOINT_PATH = f"checkpoints/{RUN_NAME}"

is_adapter = Path(f"{CHECKPOINT_PATH}/adapter_config.json").exists()
has_full_weights = Path(f"{CHECKPOINT_PATH}/model.safetensors").exists() or Path(f"{CHECKPOINT_PATH}/model.safetensors.index.json").exists()
assert is_adapter or has_full_weights, (
    f"No checkpoint found at {CHECKPOINT_PATH} after training — the training cell above must have "
    "failed. Scroll up to its output for the real error rather than continuing."
)
print("run_name:", RUN_NAME)
print("checkpoint:", CHECKPOINT_PATH, "(adapter)" if is_adapter else "(full model)")
print("contents:", sorted(os.listdir(CHECKPOINT_PATH)))

## Evaluate the resulting checkpoint

Task EM/F1 on the held-out test split + MMLU forgetting-benchmark accuracy (the fixed, committed
150-question subset — same questions as the Step 3 baseline, so the forgetting delta computed in
Step 7 is meaningful).

In [ ]:
!PYTHONPATH=. python scripts/evaluate.py --config {CONFIG} --checkpoint {CHECKPOINT_PATH}

In [ ]:
# Quick sanity-check print of this run's headline numbers before pushing.
train_results_path = f"results/{RUN_NAME}_train.json"
eval_results_path = f"results/{RUN_NAME}_eval.json"
assert Path(train_results_path).exists(), f"{train_results_path} missing — training results were not written."
assert Path(eval_results_path).exists(), f"{eval_results_path} missing — the evaluate cell above must have failed."

with open(train_results_path) as f:
    train_result = json.load(f)
with open(eval_results_path) as f:
    eval_result = json.load(f)

print("=== Training cost ===")
print("wall_clock_seconds:", train_result["wall_clock_seconds"])
print("peak_memory_mb:", train_result["peak_memory_mb"])
print("train_loss:", train_result["train_loss"], "| val_loss:", train_result["val_loss"])
print()
print("=== Task + forgetting metrics ===")
print("task EM/F1:", eval_result["task_metrics"])
print("MMLU accuracy:", eval_result["mmlu"]["accuracy"])

## Push results back to GitHub

Only the small `results/*.json` files — checkpoints stay gitignored, as already set up in the repo.

In [ ]:
!git add results/*.json
!git commit -m "Colab run: {RUN_NAME} training + eval results" || echo "(nothing new to commit)"
!git push

In [ ]:
# Confirm the push actually landed on GitHub (not just committed locally).
!git log origin/main -1 --format='Pushed OK — origin/main is now: %H %s'

## (Optional) Download the checkpoint

Only needed if you want to keep the trained weights beyond this Colab session (e.g. for the Step 9
inference endpoint). LoRA/QLoRA adapters are small (~5-15MB); the full fine-tune checkpoint is ~1-2GB
and slow to download — consider copying it to Google Drive instead if you need to keep it.

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive(f"/content/{RUN_NAME}_checkpoint", "zip", CHECKPOINT_PATH)
files.download(zip_path)